In [ ]:
from btreport.llm_report_generation.ollama_report_gen_sgem import generar_reporte_es
from btreport.utils.mls_septum import texto_reporte as texto_mls

# Metadata de ejemplo (en producción viene del pipeline completo)
metadata_ejemplo = {
    "Tumor Location": "left frontal lobe",
    "Side of Tumor Epicenter": "left",
    "Proportion Enhancing": 57.2,
    "Proportion Necrosis": 20.1,
    "Proportion of Oedema": 22.7,
    "Lesion Sizes APxTVxCC (cm)": "4.3 x 4.3 x 6.6",
    "midline_shift_present": "Yes",
    "max_shift_mm": 4.1,
    "level_max_shift": "septum pellucidum",
    # Variables SGEM
    "cisterns_status": resultado["cisterns_status"],
    "compression_ratio": resultado["compression_ratio"],
    "mls_septum_mm": 4.1,
    "mls_direccion": "izquierda→derecha",
    "mls_categoria": "leve",
}

reporte = generar_reporte_es(
    subject_id="BraTS-GLI-00000-000",
    metadata=metadata_ejemplo,
    model="llama3:8b"
)
print(reporte)

In [ ]:
import sys
sys.path.insert(0, ".")

import numpy as np
import nibabel as nib
from btreport.utils.clasificar_cisternas import clasificar_cisternas, texto_reporte

# Test con atlas real
atlas = nib.load("btreport/utils/Cistern_Segmentations.nii.gz")
data_atlas = atlas.get_fdata().astype(np.int16)
affine = atlas.affine

# Tumor simulado
tumor = np.zeros_like(data_atlas, dtype=np.int16)
tumor[70:120, 90:140, 70:120] = 2
nib.save(nib.Nifti1Image(tumor, affine), "/tmp/tumor_test.nii.gz")
nib.save(nib.Nifti1Image(data_atlas, affine), "/tmp/cisternas_test.nii.gz")

resultado = clasificar_cisternas("/tmp/cisternas_test.nii.gz", "/tmp/tumor_test.nii.gz")
print(texto_reporte(resultado))

In [ ]:
# Instalar Ollama en Colab
!curl -fsSL https://ollama.com/install.sh | sh

# Arrancar servidor en background
import subprocess, time
subprocess.Popen(["ollama", "serve"])
time.sleep(5)

# Descargar llama3:8b (~4.7GB, tarda unos minutos)
!ollama pull llama3:8b

In [ ]:
import os

# Descargar caso público de BraTS 2023
!pip install gdown -q
import gdown

# Caso de ejemplo (BraTS-GLI-00000-000)
os.makedirs("data/BraTS-GLI-00000-000", exist_ok=True)

# Alternativa: usar el dataset de HuggingFace para ver features sin imágenes
import json
with open("btreport_brats23.json") as f:
    data = json.load(f)

case_id = list(data.keys())[0]
print(f"Caso: {case_id}")
print(f"Reporte existente:\n{data[case_id]['Predicted Report (llama3:70b)'][:500]}")

In [ ]:
!git clone https://github.com/EstebanArenas1/SGEM.git
%cd SGEM
import sys
sys.path.insert(0, ".")
print("Repo SGEM clonado correctamente")

In [ ]:
!pip install nibabel numpy scipy scikit-image matplotlib \
             SynthSeg ollama sanitext datasets -q

In [ ]:
import torch
print(f"GPU disponible: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Ninguna'}")